In [ ]:
from PIL import Image
import json, math, os

cfg = json.load(open("../digitize_maps/map_config.json"))
MAPS_DIR = "../digitize_maps/maps"  

def haversine_km(lon1, lat1, lon2, lat2):
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = math.radians(lat2-lat1), math.radians(lon2-lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(a))

for key, e in cfg.items():
    img_path = os.path.join(MAPS_DIR, e["image"])
    if not os.path.exists(img_path):
        print(f"{e['source']}: image not found, skipping"); continue
    w, h = Image.open(img_path).size
    r1, r2 = e["ref_points"]
    ground_km = haversine_km(r1["lon"], r1["lat"], r2["lon"], r2["lat"])
    # diagonal pixel distance between refs isn't known without the ref pixel
    # coords, so approximate using image width as the mapped span:
    # better: use the saved pixel_x/pixel_y of the refs if you have them.
    m_per_px_x = (ground_km * 1000) / w
    # plausible click jitter of +/- 3 px and symbol radius ~ a few px:
    for jitter_px in (3, 5):
        print(f"{e['source']:<28} {w}x{h}px  ~{m_per_px_x:,.0f} m/px  "
              f"-> ±{jitter_px}px click ≈ ±{m_per_px_x*jitter_px/1000:.2f} km")

Mueller et al. 2012          1538x1168px  ~32 m/px  -> ±3px click ≈ ±0.09 km
Mueller et al. 2012          1538x1168px  ~32 m/px  -> ±5px click ≈ ±0.16 km
Gerstl et al. 2006           1326x850px  ~79 m/px  -> ±3px click ≈ ±0.24 km
Gerstl et al. 2006           1326x850px  ~79 m/px  -> ±5px click ≈ ±0.40 km
van de Bogaart et al. 2013   820x808px  ~128 m/px  -> ±3px click ≈ ±0.39 km
van de Bogaart et al. 2013   820x808px  ~128 m/px  -> ±5px click ≈ ±0.64 km
Elnaiem et al. 2003          794x698px  ~173 m/px  -> ±3px click ≈ ±0.52 km
Elnaiem et al. 2003          794x698px  ~173 m/px  -> ±5px click ≈ ±0.86 km
Hassan et al. 2020           568x782px  ~733 m/px  -> ±3px click ≈ ±2.20 km
Hassan et al. 2020           568x782px  ~733 m/px  -> ±5px click ≈ ±3.67 km


In [ ]:
import glob, os, csv, math
from itertools import combinations

OUTPUT_DIR = "../digitize_maps/outputs"  

def haversine_km(lon1, lat1, lon2, lat2):
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

files = [f for f in sorted(glob.glob(os.path.join(OUTPUT_DIR, "digitized_*.csv")))
         if "all_maps" not in f]

for fp in files:
    rows = list(csv.DictReader(open(fp)))
    if len(rows) < 2:
        print(f"{os.path.basename(fp)}: <2 points, skipping"); continue
    pts = [(float(r["pixel_x"]), float(r["pixel_y"]),
            float(r["longitude"]), float(r["latitude"])) for r in rows]
    best_px, best = 0, None
    for a, b in combinations(pts, 2):
        dpx = math.hypot(a[0]-b[0], a[1]-b[1])
        if dpx > best_px:
            best_px, best = dpx, (a, b)
    a, b = best
    m_per_px = (haversine_km(a[2], a[3], b[2], b[3]) * 1000) / best_px
    print(f"{rows[0]['source']:<28} {len(rows):>3} pts  {m_per_px:6.1f} m/px  "
          f"->  ±3px ~ {m_per_px*3/1000:.2f} km   ±5px ~ {m_per_px*5/1000:.2f} km")

Elnaiem et al. 2003           11 pts   266.0 m/px  ->  ±3px ~ 0.80 km   ±5px ~ 1.33 km
Gerstl et al. 2006             9 pts   165.3 m/px  ->  ±3px ~ 0.50 km   ±5px ~ 0.83 km
Hassan et al. 2020             4 pts   928.3 m/px  ->  ±3px ~ 2.78 km   ±5px ~ 4.64 km
Mueller_et_al_2012_Fig2       33 pts    78.9 m/px  ->  ±3px ~ 0.24 km   ±5px ~ 0.39 km
van de Bogaart et al. 2013     5 pts   814.5 m/px  ->  ±3px ~ 2.44 km   ±5px ~ 4.07 km
